# PROJE: İSTANBUL TAKSİ DAĞILIMINDAKİ DENGESİZLİK VE STOKASTİK ANALİZİ

Hipotez 1 (Dengesiz Yığılma): Simülasyonu "sonsuz" döngüye (yeterince uzun adım, örn: 10.000) sokup, sistemin oturduğu son dağılımı kaydedeceğiz.

Hipotez 2 (Geri Dönüş Süresi): Kodun içine bir sayaç ekleyeceğiz. Bir taksi "Beylikdüzü"ne geldiği anı kaydedip, bir dahaki gelişine kadar geçen süreyi sayacağız.

Hipotez 3 (Trafik Tuzağı): Matrisi manipüle edeceğiz. Köprü geçiş olasılıklarını (Avrupa <-> Anadolu) yapay olarak düşürüp, "Kriz Senaryosu"nu çalıştıracağız ve normal durumla farkına bakacağız.

In [1]:
using Statistics
import Random

In [2]:
semtler = ["Besiktas", "Kadikoy", "Sisli", "Beylikduzu", "Umraniye"]
# Semtlerin Yakaları: 1=Avrupa, 2=Anadolu 
yakalar = [1, 2, 1, 1, 2] 

# GEÇİŞ MATRİSİ (P) - NORMAL DURUM
# Bu matris, bir taksinin "şu anki" konumundan "bir sonraki" konuma geçme olasılığını gösterir.
# Satırlar: Şu anki konum, Sütunlar: Gidilecek konum
P_normal = [
    0.50   0.20   0.25   0.02   0.03; # Beşiktaş
    0.15   0.60   0.15   0.05   0.05; # Kadıköy
    0.30   0.10   0.55   0.02   0.03; # Şişli
    0.05   0.05   0.60   0.20   0.10; # Beylikdüzü
    0.10   0.30   0.20   0.05   0.35  # Ümraniye
]

# Gerçek Hayattaki Yolcu Talebi (Varsayımsal Nüfus Yoğunluğu %)
gercek_talep = [0.20, 0.25, 0.20, 0.25, 0.10] 



5-element Vector{Float64}:
 0.2
 0.25
 0.2
 0.25
 0.1

# TERS DÖNÜŞÜM YÖNTEMİ 

In [3]:

# 0-1 arası rastgele üretilen 'u' sayısını, kümülatif olasılıkları kullanarak ilgili semte dönüştürür.
function sonraki_semti_sec(suanki_semt_idx, matris)
    u = rand() # 0 ile 1 arasında düzgün dağılım 
    kumulatif = 0.0
    satir_olasiliklari = matris[suanki_semt_idx, :]
    
    for i in 1:length(satir_olasiliklari)
        kumulatif += satir_olasiliklari[i]
        # Eğer u kümülatif toplamın altındaysa o semti seç
        if u <= kumulatif
            return i
        end
    end
    return length(satir_olasiliklari)
end



sonraki_semti_sec (generic function with 1 method)

# MONTE CARLO

In [4]:
# Verilen matris ve taksi sayısı ile sistemi 'n_adim' kadar koşturur.
# Hem dağılımı hem de (istenirse) geri dönüş sürelerini hesaplar.
function simulasyonu_calistir(matris, n_taksi, n_adim, takip_edilecek_semt_idx=nothing)
    n_semt = size(matris, 1)
    taksi_konumlari = rand(1:n_semt, n_taksi) # Başlangıçta taksileri rastgele dağıt
    
    # Geri Dönüş Süresi değişkenleri
    son_gorulme_zamani = zeros(Int, n_taksi) 
    donus_sureleri = Int[] 
    
    for t in 1:n_adim
        for i in 1:n_taksi
            eski_konum = taksi_konumlari[i]
            yeni_konum = sonraki_semti_sec(eski_konum, matris)
            taksi_konumlari[i] = yeni_konum
            
            # Hipotez 2 için veri toplama:
            # Eğer taksi hedef semte geldiyse, en son gelişinden bu yana ne kadar geçti?
            if takip_edilecek_semt_idx !== nothing && yeni_konum == takip_edilecek_semt_idx
                if son_gorulme_zamani[i] != 0
                    gecen_sure = t - son_gorulme_zamani[i]
                    push!(donus_sureleri, gecen_sure)
                end
                son_gorulme_zamani[i] = t
            end
        end
    end
        # Sonuç dağılımını (Stationary Distribution) hesapla
    dagilim_sayilari = zeros(Int, n_semt)
    for k in taksi_konumlari
        dagilim_sayilari[k] += 1
    end
    dagilim_oranlari = dagilim_sayilari ./ n_taksi
    
    return dagilim_oranlari, donus_sureleri
end
 

simulasyonu_calistir (generic function with 2 methods)

# 3. HİPOTEZ TESTLERİ VE SONUÇLAR

# HİPOTEZ 1: DENGESİZ YIĞILMA 

Soru: Sistem kendi haline bırakıldığında,taksiler yolcu talebinin olduğu yerlerde mi, yoksa sistemin onları hapsettiği yerlerde mi birikiyor?


In [5]:
println(">> TEST 1: Dengesiz Yığılma Analizi (Arz-Talep Farkı)")

n_taksi = 1000
n_adim = 2000 # Sistemin kararlı duruma geçmesi için yeterli adım
sonuc_dagilim, _ = simulasyonu_calistir(P_normal, n_taksi, n_adim)

println("Semt         | Talep (%) | Arz (%) | Durum")
println("----------------------------------------------")
for i in 1:length(semtler)
    talep_yuzde = round(gercek_talep[i] * 100, digits=1)
    arz_yuzde = round(sonuc_dagilim[i] * 100, digits=1)
    fark = sonuc_dagilim[i] - gercek_talep[i]
    
    durum = "Dengeli"
    if fark < -0.05
        durum = "KRİZ (Taksi Yok)"
    elseif fark > 0.05
        durum = "YIĞILMA (Boş)"
    end
    
    println(rpad(semtler[i], 12), " | ", rpad(talep_yuzde, 9), " | ", rpad(arz_yuzde, 7), " | ", durum)
end
println("\n")




>> TEST 1: Dengesiz Yığılma Analizi (Arz-Talep Farkı)
Semt         | Talep (%) | Arz (%) | Durum
----------------------------------------------
Besiktas     | 20.0      | 30.1    | YIĞILMA (Boş)
Kadikoy      | 25.0      | 26.2    | Dengeli
Sisli        | 20.0      | 35.2    | YIĞILMA (Boş)
Beylikduzu   | 25.0      | 3.0     | KRİZ (Taksi Yok)
Umraniye     | 10.0      | 5.5     | Dengeli




# HİPOTEZ 2: GERİ DÖNÜŞ SÜRESİ 


 Soru: Uzak bir semtten (Örn: Beylikdüzü) ayrılan bir taksinin, sistemin doğal akışıyla tekrar o semte dönmesi ortalama kaç adım sürer?
 Eğer süre çok uzunsa, o semt bir "Taksi Kara Deliği"dir.

In [6]:
target_semt = 4 # Beylikdüzü
println(">> TEST 2: Geri Dönüş Süresi (", semtler[target_semt], " İçin)")

_, donus_sureleri = simulasyonu_calistir(P_normal, 500, 2000, target_semt)
ortalama_donus = round(mean(donus_sureleri), digits=1)

println("Analiz: Bir taksi ", semtler[target_semt], "'nden ayrıldıktan sonra tekrar gelmesi" ,ortalama_donus, "adım(sefer) sürüyor ")

if ortalama_donus > 20
    println("Yorum: Hipotez doğrulandı. Süre çok uzun, taksiler buraya geri dönemiyor.")
else
    println("Yorum: Süre makul, dengeli.")
end
println("\n")




>> TEST 2: Geri Dönüş Süresi (Beylikduzu İçin)
Beylikduzu'nden ayrıldıktan sonra tekrar gelmesi27.0adım(sefer) sürüyor 
Yorum: Hipotez doğrulandı. Süre çok uzun, taksiler buraya geri dönemiyor.




# HİPOTEZ 3: TRAFİK TUZAĞI VE VERİMSİZLİK 

 Soru: Eğer köprü trafiği yüzünden yakalar arası geçiş zorlaşırsa (%80 azalırsa),taksi dağılımı nasıl bozulur? Taksiler kendi yakalarında hapsolur mu?

In [7]:
println(">> TEST 3: Trafik Tuzağı (Köprüler Tıkanırsa Ne Olur?)")

# Kriz Matrisi Oluşturma: Yakalar arası geçiş olasılıklarını düşür (Bias ekle)
P_kriz = copy(P_normal)
ceza_katsayisi = 0.20 # Geçiş ihtimalini %80 azalt 

for i in 1:5 
    for j in 1:5
        if yakalar[i] != yakalar[j] # Farklı yakalar ise ceza uygula
            P_kriz[i, j] *= ceza_katsayisi
        end
    end
    # Matrisi normalize et 
    toplam = sum(P_kriz[i, :])
    P_kriz[i, :] = P_kriz[i, :] ./ toplam
end

sonuc_kriz, _ = simulasyonu_calistir(P_kriz, n_taksi, n_adim)

println("Semt         | Normal (%) | Kriz (%) | Değişim")
println("----------------------------------------------")
for i in 1:length(semtler)
    normal_yuzde = round(sonuc_dagilim[i] * 100, digits=1)
    kriz_yuzde = round(sonuc_kriz[i] * 100, digits=1)
    degisim = round((sonuc_kriz[i] - sonuc_dagilim[i]) * 100, digits=1)
    
    println(rpad(semtler[i], 12), " | ", rpad(normal_yuzde, 10), " | ", rpad(kriz_yuzde, 8), " | ", degisim)
end

>> TEST 3: Trafik Tuzağı (Köprüler Tıkanırsa Ne Olur?)
Semt         | Normal (%) | Kriz (%) | Değişim
----------------------------------------------
Besiktas     | 30.1       | 35.0     | 4.9
Kadikoy      | 26.2       | 22.5     | -3.7
Sisli        | 35.2       | 35.2     | 0.0
Beylikduzu   | 3.0        | 2.9      | -0.1
Umraniye     | 5.5        | 4.4      | -1.1
